# Advanced Analytics - Bluestock Mutual Fund Analytics

This notebook covers Historical VaR/CVaR, rolling Sharpe ratio, investor cohorts, SIP continuity, risk-based recommendations, and sector concentration using HHI.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DB_PATH = Path("../bluestock_mf.db")
REPORTS_DIR = Path("../reports")
CHART_DIR = REPORTS_DIR / "charts"

conn = sqlite3.connect(DB_PATH)
fund = pd.read_sql("SELECT * FROM dim_fund", conn)
perf = pd.read_sql("SELECT * FROM fact_performance", conn)
nav = pd.read_sql("""
SELECT n.amfi_code, d.date, n.nav
FROM fact_nav n
JOIN dim_date d ON n.date_id = d.date_id
""", conn)
txn = pd.read_sql("""
SELECT t.*, d.date AS transaction_date
FROM fact_transactions t
JOIN dim_date d ON t.transaction_date_id = d.date_id
""", conn)
holdings = pd.read_sql("SELECT * FROM portfolio_holdings", conn)
conn.close()

nav["date"] = pd.to_datetime(nav["date"])
txn["transaction_date"] = pd.to_datetime(txn["transaction_date"])
nav = nav.sort_values(["amfi_code", "date"])
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()
nav.head()


## Historical VaR and CVaR

Historical 95% VaR was calculated as the 5th percentile of daily returns, and CVaR was calculated as the mean of returns below that threshold.

In [ ]:
pd.read_csv(REPORTS_DIR / "var_cvar_report.csv").sort_values("var_95_daily").head(10)

## Rolling 90-day Sharpe

Rolling Sharpe uses a 90-day rolling mean divided by 90-day rolling standard deviation, annualized by √252.

In [ ]:
from IPython.display import Image
Image(filename=str(CHART_DIR / "rolling_sharpe_chart.png"))

## Investor cohort analysis

Investor cohorts were grouped by first transaction year to compare average SIP amount, total invested amount, and preferred fund.

In [ ]:
pd.read_csv(REPORTS_DIR / "investor_cohort_analysis.csv")

## SIP continuity analysis

Investors with 6 or more SIP transactions were evaluated by average gap between SIP dates; average gap above 35 days is flagged as at-risk.

In [ ]:
pd.read_csv(REPORTS_DIR / "sip_continuity_analysis.csv").head(10)

## Risk-based recommender

The simple recommender filters funds by risk appetite and returns the top 3 funds ranked by Sharpe ratio.

In [ ]:
import sys
sys.path.append("../scripts")
from recommender import recommend_funds
recommend_funds("Moderate")

## Sector HHI concentration

HHI was calculated as Σ(weight²) using sector weights for equity fund portfolio holdings. Higher HHI means greater concentration.

In [ ]:
pd.read_csv(REPORTS_DIR / "sector_hhi_concentration.csv").sort_values("hhi", ascending=False).head(10)

## 5 Advanced EDA Insights

1. ABSL Small Cap Fund - Regular - Growth has the lowest 95% daily VaR at -2.39%, making it the largest downside-tail-risk fund in the dataset.
2. The 2024 investor cohort contributes the highest total invested value at ₹3,491,125,187.
3. SIP continuity among investors with 6+ SIP transactions is 2.2% regular, while 97.8% are flagged at-risk using the 35-day average-gap rule.
4. Axis Bluechip Fund - Regular - Growth has the highest sector HHI at 0.206, indicating the most concentrated equity portfolio among available holdings.
5. ICICI Pru Liquid Fund - Regular - Growth ranks highest by Sharpe ratio at 7.68, making it the strongest risk-adjusted performer in the performance table.